# 词向量

前面几章，我们依次解决了三个问题：

- **单热编码**：把单个词变成无大小关系的向量；
- **词袋**：把一整句话（多个词）压缩成一个固定向量；
- **嵌入**：把高维稀疏的单热向量压缩成低维稠密的向量。

嵌入层的权重是通过训练学到的，也就是说，每个词对应的嵌入向量里，隐含了模型在数据中发现的某种规律。打个比方：单热编码就像给每种花起个**名字**：名字只是区分符号，不包含任何信息；而嵌入向量更像是给每种花写的**简介**：不仅可以区分，还告诉你花的特点。

上一章，我们通过情感分类任务训练嵌入，学到的是每个词在情感上的特征（偏正面还是偏负面）。本章我们将引入一种新的嵌入思路：**词向量**（Word Vector），一种专门为捕捉相互语义关系而设计的嵌入向量。

---

## Word2Vec

2013 年，Google 的 托马斯（Tomas）团队提出了 **Word2Vec**，它基于一个简单的假设，**分布假设**（Distributional Hypothesis）：在相似上下文中出现的词，往往有相似的语义。

比如“猫”和“狗”经常出现在类似的句子里（“我养了一只___”、“___喜欢玩”），所以它们的词向量应该相近。

Word2Vec 的核心思路是**用文字预测文字**，而不是用额外的评价来预测。有两种具体做法：

| 方法 |  输入  |   预测目标    |                   直觉 |
|------|:----:|:---------:|---------------------|
| **CBOW**（连续词袋） | 上下文  |    中心词    |      填空题：根据周围的词猜中间的词 |
| **Skip-gram**（跳字） | 中心词  |    上下文    |  思维导图：从一个词想到周围可能出现的词 |

以"老鼠爱大米"为例，取窗口大小为 2（中心词两侧各 2 个词）：

- **CBOW**：输入"老"、"鼠"、"大"、"米"，预测"爱"；
- **Skip-gram**：输入"爱"，分别预测"老"、"鼠"、"大"、"米"。

本章我们来实现 **CBOW 连续词袋**算法。

In [1]:
import csv
import re
from abc import abstractmethod, ABC

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __init__(self):
        self.training = True

    def __call__(self, x: Tensor):
        return self.forward(x)

    def train(self):
        self.training = True

    def eval(self):
        self.training = False

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### IMDB 数据集

我们继续使用 TinyIMDB 数据集，但这次只用影评文字，不用情感标签。

**CBOW 训练数据的构建方式**：对每条影评，用一个长度为 `sequence_length` 的滑动窗口扫描，每次截取连续的 `sequence_length` 个词。其中**中间位置**的词作为标签（要预测的词），其余位置的词组成词袋作为特征。

以 `sequence_length=5` 为例，窗口中有 5 个词，中间位置（索引 2）是要预测的中心词，两侧各 2 个词是上下文：

```
句子：  the  film  was  brilliant  and
索引：   0    1    2       3        4

特征（上下文）：[the, film, brilliant, and]  →  词袋索引列表
标签（中心词）：was                          →  单热编码向量
```

窗口每次向右滑动一格，直到扫完整条影评，每条影评可以生成 `len(tokens) - sequence_length + 1` 个训练样本。

标签是中心词的**单热编码**（长度等于词表大小），对应一个多分类问题：模型需要从整个词表中猜出最可能的中心词。

In [8]:
class IMDBDataset(Dataset):

    def __init__(self, filename, sequence_length=5):
        self.filename = filename
        self.sequence_length = sequence_length
        super().__init__()

    def load(self):
        self.reviews = []
        self.sentiments = []
        with open(self.filename, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)
            for row in reader:
                self.reviews.append(self.clean_text(row[0].lower()).split())
                self.sentiments.append([0 if row[1] == "negative" else 1])

        self.vocabulary = sorted(set(word for line in self.reviews for word in line))
        self.word2index = {word: index for index, word in enumerate(self.vocabulary)}
        self.index2word = {index: word for index, word in enumerate(self.vocabulary)}
        self.tokens = [[self.word2index[word] for word in line if word in self.word2index] for line in self.reviews]

        # 生成词序列
        self.train_features, self.train_labels = self.create_sequence(self.tokens[:-100])
        self.test_features, self.test_labels = self.create_sequence(self.tokens[-100:])

    @staticmethod
    def clean_text(text):
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        return text

    # CBOW：滑动窗口生成（上下文词袋, 中心词单热编码）训练对
    def create_sequence(self, tokens):
        # 中心词在窗口中的位置
        center = self.sequence_length // 2
        features, labels = [], []
        for line in tokens:
            for start in range(len(line) - self.sequence_length + 1):
                window = line[start: start + self.sequence_length]
                # 上下文：窗口中除中心词外的所有词的索引编码
                context = [window[i] for i in range(self.sequence_length) if i != center]
                features.append(context)
                # 标签：中心词的单热编码
                labels.append(self.onehot(window[center]))
        return features, labels

    def encode(self, text):
        words = self.clean_text(text.lower()).split()
        return [self.word2index[word] for word in words if word in self.word2index]

    def decode(self, tokens):
        return " ".join([self.index2word[index] for index in tokens])

    def onehot(self, index):
        vector = np.zeros(len(self.vocabulary))
        vector[index] = 1
        return vector

    @staticmethod
    def argmax(vector):
        return np.argmax(vector)

    def estimate(self, predictions):
        labels = np.array(self.labels)
        correct = (np.argmax(predictions.data, axis=1) == np.argmax(labels, axis=1)).sum()
        return correct / len(labels)

## 模型

### 线性层

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.random.randn(out_size, in_size) * np.sqrt(2 / in_size))
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 顺序层

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def train(self):
        super().train()
        for l in self.layers:
            l.train()

    def eval(self):
        super().eval()
        for l in self.layers:
            l.eval()

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

    def __repr__(self):
        return '\n'.join(str(l) for l in self.layers if str(l))

### 嵌入层

我们对嵌入层进行了一些小小的改动：添加了一个新的参数：`axis`。

* `axis = 1`：词袋模式，嵌入向量压缩到一个向量，形状是`（1, embedding_size）`；
* `axis = None`：词向量模式，保持每个单词单独的嵌入向量，形状是`（word_count, embedding_size）`。

In [11]:
class Embedding(Layer):

    def __init__(self, vocabulary_size, embedding_size, axis=None):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.embedding_size = embedding_size
        self.axis = axis
        self.weight = Tensor(np.random.randn(vocabulary_size, embedding_size) * np.sqrt(2 / vocabulary_size))

    def forward(self, x: Tensor):
        w = self.weight.data[x.data]
        p = Tensor(np.sum(w, axis=self.axis) if self.axis is not None else w)

        def gradient_fn():
            np.add.at(self.weight.grad, x.data, p.grad)

        p.gradient_fn = gradient_fn
        p.parents = {self.weight}
        return p

    @property
    def parameters(self):
        return [self.weight]

    def __repr__(self):
        return f'Embedding[weight{self.weight.shape}; vocabulary={self.vocabulary_size}; embedding={self.embedding_size}]'

### 展平层

CBOW 的嵌入层输出形状是 `(word_count, embedding_size)`，而线性层需要一个一维向量作为输入。我们用**展平层**（Flatten）把多维张量「压平」成一行，形状从 `(4, 32)` 变成 `(1, 128)`。

这里 128 = 4（上下文词数）× 32（嵌入维度），所以 `Linear` 的输入维度是 128。

In [12]:
class Flatten(Layer):

    def forward(self, x):
        p = Tensor(x.data.reshape(x.data.shape[0], -1))

        def gradient_fn():
            x.grad += p.grad.reshape(x.data.shape)

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

### 交叉熵损失函数

对应多分类问题，我们使用**交叉熵损失**（Cross-Entropy Loss，CE）。

In [13]:
class CELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        log = np.log(np.clip(softmax, 1e-10, 1))
        ce = Tensor(0 - np.sum(y.data * log) / len(y.data))

        def gradient_fn():
            p.grad += (softmax - y.data) / len(y.data)

        ce.gradient_fn = gradient_fn
        ce.parents = {p}
        return ce

### 优化器（随机梯度下降）

In [14]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

### 神经网络模型

In [15]:
class NNModel(Model):

    def train(self, dataset, epochs):
        self.layer.train()
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.reset()
                predictions = self.layer(features)
                loss = self.loss_fn(predictions, labels)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        self.layer.eval()
        dataset.eval()

        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 训练

### 超参数：学习率

In [16]:
LEARNING_RATE = 0.1

### 超参数：轮次

In [17]:
EPOCHS = 100

### 序列长度

我们增加了一个新的超参数：**序列长度**（Sequence Length）。用来定义每次迭代从数据集截取多长的文字作为输入。

In [18]:
SEQUENCE_LENGTH = 5

### 建模

模型结构：

- **嵌入层** `Embedding(150, 32)`：把 4 个上下文词的索引分别查表，得到 `(4, 32)` 的嵌入矩阵；
- **展平层** `Flatten()`：把 `(4, 32)` 压平成 `(1, 128)`；
- **线性层** `Linear(128, 150)`：输入 128 维，输出 150 维（词表大小），每个维度对应一个词的得分；
- 损失函数 `CELoss` 内含 Softmax，把 150 维 logits 转成概率，与单热标签计算交叉熵。

这里只训练 10 轮（相比情感分类的 100 轮），因为 CBOW 的训练样本数量更多（每条影评生成多个样本），10 轮就已经有相当训练量。

In [19]:
dataset = IMDBDataset('tinyimdb.csv', sequence_length=SEQUENCE_LENGTH)

layer = Sequential([
    Embedding(len(dataset.vocabulary), 32),
    Flatten(),
    Linear((SEQUENCE_LENGTH - 1) * 32, len(dataset.vocabulary))
])
loss_fn = CELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(layer)

Embedding[weight(150, 32); vocabulary=150; embedding=32]
Flatten[]
Linear[weight(150, 128); bias(150,)]


### 迭代

In [20]:
model.train(dataset, EPOCHS)

## 验证

### 测试

在测试集上评估中心词预测准确率：

In [21]:
predictions, loss = model.test(dataset)
accuracy = dataset.estimate(predictions)
print(f'accuracy: {accuracy:.2%}')

accuracy: 66.49%


66.5% 的准确率看起来不高，但这是一个从 150 个词里选一个的多分类任务：随机猜的准确率只有 1/150 ≈ 0.7%。能达到 65% 说明模型确实学到了上下文与中心词之间的关联。

更重要的是，Word2Vec 的真正价值不在于预测准确率，而在于训练完成后**嵌入层的权重**，那就是我们需要的词向量。

### 预测对比

我们来看几条具体的预测结果，直观感受模型学到了什么：

In [22]:
# 展示前 10 条预测结果
for i in range(min(10, len(predictions.data))):
    context_words = dataset.decode(dataset.features[i])
    label_word = dataset.decode([dataset.argmax(dataset.labels[i])])
    pred_word = dataset.decode([dataset.argmax(predictions.data[i])])
    correct = '✓' if label_word == pred_word else '✗'
    print(f'{correct} context: [{context_words}]  label: {label_word:12s}  prediction: {pred_word}')

✓ context: [horrible film soundtrack and]  label: the           prediction: the
✗ context: [film the and visual]  label: soundtrack    prediction: acting
✓ context: [the soundtrack visual effects]  label: and           prediction: and
✓ context: [soundtrack and effects were]  label: visual        prediction: visual
✓ context: [and visual were weak]  label: effects       prediction: effects
✓ context: [visual effects weak i]  label: were          prediction: were
✗ context: [effects were i hated]  label: weak          prediction: pathetic
✓ context: [were weak hated the]  label: i             prediction: i
✗ context: [weak i the plot]  label: hated         prediction: disliked
✓ context: [great movie screenplay and]  label: the           prediction: the


### 近似词

让我们观察一下权重最相似，和最不相似的词向量之间有什么特点。

In [23]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)


def find_similar_words(word, top_n=5):
    if word not in dataset.word2index:
        return

    idx = dataset.word2index[word]
    target_vec = layer.layers[0].weight.data[idx]

    similarities = []
    for i, w in enumerate(dataset.vocabulary):
        if i == idx:
            continue
        sim = cosine_similarity(target_vec, layer.layers[0].weight.data[i])
        similarities.append((sim, w))

    similarities.sort(reverse=True)
    print(f'Similar to "{word}":')
    for sim, w in similarities[:top_n]:
        print(f'  {w:20s}  {sim:.4f}')
    print(f'Not silimar to "{word}": ')
    for sim, w in similarities[-top_n:][::-1]:
        print(f'  {w:20s}  {sim:.4f}')


find_similar_words('good')

Similar to "good":
  great                 0.4815
  fantastic             0.4205
  best                  0.3772
  effects               0.3764
  editing               0.3506
Not silimar to "good": 
  again                 -0.3727
  time                  -0.3546
  very                  -0.3367
  us                    -0.3257
  my                    -0.3204


可以看出，Word2Vec 认为“相似”的词并不全是纯粹的同义词，而是在上下文关系中可以相互替代的词。

这与情感分类训练出的嵌入不同。情感分类任务的嵌入区分的是**正面词 vs 负面词**；Word2Vec 的嵌入捕捉的是**上下文分布的相似性**，语义关系更丰富，也更复杂。

## 课后练习

本章实现了 CBOW（用上下文预测中心词）。现在试着修改 `create_sequence()` 方法，改为 **Skip-gram** 策略：

- **输入**：中心词的索引（列表形式，只含 1 个元素）；
- **标签**：上下文词的单热编码（每个上下文词单独生成一条训练样本）。

以 `sequence_length=5` 的窗口 **the film was brilliant and** 为例，Skip-gram 应生成 4 条训练样本：

| 输入（中心词） | 标签（上下文词） |
|:--------------:|:----------------:|
| was | the |
| was | film |
| was | brilliant |
| was | and |